# Demo 30: Dyna-Q: Real and Imagined Experience

**挿入位置：Dyna・モデルベース学習のスライド**　｜　**実行環境：Google Colab**

1回の実環境経験から、Q値を直接更新すると同時に**状態遷移モデル**を学習し、モデルで作った仮想経験から追加学習するDyna-Qです。

**使い方**：ランタイムで「すべてのセルを実行」した後、スライダー／ボタンで条件を変更します。グラフ中の説明は英語、手順と解説は日本語です。CPUで動作します（Demo 28/29 はPyTorchを使用）。

**報酬の添字**：状態 $s_t$ で行動 $a_t$ を実行した結果得る報酬を $r_t$ と呼び、次状態を $s_{t+1}$ とします。経験は $(s_t,a_t,r_t,s_{t+1})$ です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
# All maze-based demos share the same 5x5 world and conventions.
# row 0 = top, column 0 = left. The shortest path S->G has 8 steps.
N=5
START=(4,0); GOAL=(0,4)
WALLS=frozenset({(1,1),(2,1),(2,3),(3,3),(3,4)})
ACTIONS=[(-1,0),(1,0),(0,-1),(0,1)]
ACTION_NAMES=['Up','Down','Left','Right']
STATES=[(i,j) for i in range(N) for j in range(N) if (i,j) not in WALLS]
NONTERMINAL=[s for s in STATES if s!=GOAL]

def transition(s,a, step_cost=-0.04, goal_reward=1.0, walls=WALLS):
    if s==GOAL:return GOAL,0.0,True
    di,dj=ACTIONS[a];ns=(s[0]+di,s[1]+dj)
    if not(0<=ns[0]<N and 0<=ns[1]<N) or ns in walls:ns=s
    done=ns==GOAL
    return ns, goal_reward if done else step_cost, done

def render_maze(ax,values=None,policy=None,route=None,title='',walls=WALLS,fmt='.2f'):
    from matplotlib.patches import Rectangle
    ax.set_xlim(0,N);ax.set_ylim(N,0);ax.set_aspect('equal');ax.axis('off')
    valid=[v for s,v in (values or {}).items() if s not in walls and np.isfinite(v)]
    lo=min(valid) if valid else 0;hi=max(valid) if valid else 1
    for i in range(N):
        for j in range(N):
            s=(i,j)
            if s in walls:color='#626B7B'
            elif s==GOAL:color='#FAD9CA'
            elif s==START:color='#CDECFD'
            elif values is not None and s in values and hi>lo:
                color=plt.cm.YlGnBu(0.15+0.58*(values[s]-lo)/(hi-lo))
            else:color='#FFFFFF'
            ax.add_patch(Rectangle((j,i),1,1,facecolor=color,edgecolor='#8392A6',linewidth=1.1))
            if s in walls:continue
            if s==GOAL:label='G'
            elif values is not None and s in values:label=f'{values[s]:{fmt}}'
            elif s==START:label='S'
            else:label=''
            if label:ax.text(j+.5,i+.49,label,ha='center',va='center',color='#102E57',fontsize=9 if values is not None else 13,fontweight='bold')
            if policy is not None and s in policy and s!=GOAL and values is None:
                ax.text(j+.5,i+.72,'↑↓←→'[policy[s]],ha='center',va='center',color='#1763A8',fontsize=13)
    if route and len(route)>1:
        xx=[s[1]+.5 for s in route];yy=[s[0]+.5 for s in route]
        ax.plot(xx,yy,lw=3,color='#F39B2F',marker='o',markersize=4,alpha=.92)
    ax.set_title(title,fontweight='bold',fontsize=12)

In [ ]:
def dyna_q(episodes=200,planning=5,seed=0,alpha=.2,gamma=.9,epsilon=.15,cap=100):
    rng=np.random.default_rng(seed);Q={s:np.zeros(4) for s in STATES};model={}
    steps=[];returns=[];success=[];model_sizes=[];total_real=0;total_sim=0
    for ep in range(episodes):
        s=START;score=0;done=False
        for t in range(cap):
            if rng.random()<epsilon:a=int(rng.integers(4))
            else:a=int(rng.choice(np.flatnonzero(np.isclose(Q[s],Q[s].max()))))
            ns,r,done=transition(s,a);score+=r
            target=r+(0 if done else gamma*np.max(Q[ns]));Q[s][a]+=alpha*(target-Q[s][a])
            total_real+=1;model[(s,a)]=(ns,r,done)
            keys=list(model)
            for k in range(planning):
                ss,aa=keys[int(rng.integers(len(keys)))];nn,rr,dd=model[(ss,aa)]
                y=rr+(0 if dd else gamma*np.max(Q[nn]));Q[ss][aa]+=alpha*(y-Q[ss][aa]);total_sim+=1
            s=ns
            if done:break
        steps.append(t+1);returns.append(score);success.append(int(done));model_sizes.append(len(model))
    return Q,np.array(steps),np.array(returns),np.array(success),np.array(model_sizes),total_real,total_sim

for p in [0,5,20]:
    q,steps,ret,win,sz,real,sim=dyna_q(episodes=10,planning=p)
    assert sim==p*real
print('Dyna-Q checks passed.')

In [ ]:
episodes=widgets.IntSlider(value=200,min=50,max=500,step=50,description='Episodes',continuous_update=False)
seed=widgets.IntSlider(value=5,min=0,max=100,description='Seed')
repeat=widgets.IntSlider(value=5,min=1,max=12,description='Seeds',continuous_update=False)
button=widgets.Button(description='Compare planning steps',button_style='primary');out=widgets.Output()
def run(_):
    with out:
        clear_output(wait=True)
        fig,axs=plt.subplots(1,2,figsize=(12,4))
        for planning in (0,5,20):
            arrays=[];wins=[]
            for runid in range(repeat.value):
                q,st,rew,success,m,real,sim=dyna_q(episodes.value,planning=planning,seed=seed.value+runid)
                arrays.append(st);wins.append(success)
            arrays=np.array(arrays);wins=np.array(wins);k=min(25,episodes.value//3)
            avg_steps=arrays.mean(axis=0);avg_wins=wins.mean(axis=0)
            x=np.arange(k,episodes.value+1)
            axs[0].plot(x,np.convolve(avg_steps,np.ones(k)/k,'valid'),label=f'{planning} planning')
            axs[1].plot(x,np.convolve(avg_wins,np.ones(k)/k,'valid'),label=f'{planning} planning')
            print(f'planning={planning:2}: last 30 mean steps={arrays[:,-30:].mean():.2f}, success={wins[:,-30:].mean():.2f}')
        axs[0].set(title='Real steps per episode',xlabel='Episode',ylabel='Steps (lower is better)')
        axs[1].set(title='Success rate',xlabel='Episode',ylabel='Success',ylim=(0,1))
        for a in axs:a.grid(alpha=.2);a.legend()
        plt.tight_layout();plt.show()
        print('A planning update uses a learned (s,a) -> (s_next,reward,done) model; no new real step.')
button.on_click(run)
display(widgets.VBox([widgets.HBox([episodes,repeat,seed]),button,out]));run(None)

### Dynaの1ステップ
1. 実環境で行動し、$(s_t,a_t,r_t,s_{t+1})$を取得。
2. 実経験でQ値を更新（Model-free）。
3. $(s,a)ightarrow(s',r)$の経験モデルを更新（Model learning）。
4. 既知の経験状態行動をモデル内でサンプリングして、追加のQ更新（Planning）。

今回の環境は決定的なので、モデルに最後に観測した遷移結果を保存しています。確率的な環境では遷移確率モデルの学習が必要になります。

## Try it yourself / 実験課題
- planning=0/5/20で、同じ実環境試行回数に対する成功率や移動数を比較する。
- 実経験とモデル内更新の回数がどう違うか説明する。
- モデルが不正確な場合に仮想経験が逆効果になる理由を考える。